# Lightweight Brain Tumor Classification with HOG Features and Class-Weighted SVM

**Reproduction of:** Warsito et al., *"Lightweight Brain Tumor Classification with HOG Features and Class-Weighted SVM"*, JADS 7(1), 2026.

**Pipeline**
1. Image → grayscale → (optional CLAHE, our extension) → 16×16 → HOG features
2. PCA keeping 98% of the variance (fitted on the training set only)
3. Train the SVMs: a grid search (Table 1) plus the 6 models of Table 2
4. Evaluate on the test set

## 0. Setup and dataset

In [ ]:
import os

import cv2
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from skimage.feature import hog
from sklearn.decomposition import PCA
from sklearn.metrics import (accuracy_score, classification_report, confusion_matrix,
                             ConfusionMatrixDisplay, precision_recall_fscore_support,
                             roc_auc_score)
from sklearn.model_selection import GridSearchCV
from sklearn.svm import SVC

SEED = 42
DATA_DIR = "data"

# Brain Tumor Classification (MRI) dataset: Training/ and Testing/ with one folder per class
if not os.path.isdir(DATA_DIR):
    !git clone --depth 1 -q https://github.com/SartajBhuvaji/Brain-Tumor-Classification-DataSet.git {DATA_DIR}

## 1. Features (paper Sec. 3.2–3.3)

HOG settings: 9 orientations, 2×2 pixels per cell, 3×3 cells per block, L2-Hys normalization. That gives 2916 features per image.

In [ ]:
# Folder names in the dataset. Their position = label number (0, 1, 2, 3), same as the paper.
CLASSES = ["glioma_tumor", "meningioma_tumor", "no_tumor", "pituitary_tumor"]


def preprocess(image, use_clahe):
    """Grayscale image -> (optional CLAHE) -> 16x16 -> values in 0-1."""
    if use_clahe:
        # Our extension: boost local contrast at full resolution, before shrinking
        image = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8, 8)).apply(image)
    return cv2.resize(image, (16, 16)) / 255.0


def extract_hog(image):
    """HOG with the paper's settings, plus the paper's extra /255 scaling."""
    return hog(image, orientations=9, pixels_per_cell=(2, 2), cells_per_block=(3, 3),
               block_norm="L2-Hys") / 255.0


def load_dataset(folder, use_clahe=False):
    """Read every image in folder/<class_name>/ and return (features, labels)."""
    features, labels = [], []
    for label, class_name in enumerate(CLASSES):
        class_folder = os.path.join(folder, class_name)
        for file_name in sorted(os.listdir(class_folder)):
            image = cv2.imread(os.path.join(class_folder, file_name), cv2.IMREAD_GRAYSCALE)
            if image is not None:  # skip files that are not images
                features.append(extract_hog(preprocess(image, use_clahe)))
                labels.append(label)
    return np.array(features), np.array(labels)

A single sample image, shown before and after preprocessing:

In [ ]:
folder = os.path.join(DATA_DIR, "Training", CLASSES[0])
image = cv2.imread(os.path.join(folder, sorted(os.listdir(folder))[0]), cv2.IMREAD_GRAYSCALE)
views = {"original": image, "16x16 baseline": preprocess(image, False), "16x16 CLAHE": preprocess(image, True)}

fig, axes = plt.subplots(1, 3, figsize=(9, 3))
for ax, (title, img) in zip(axes, views.items()):
    ax.imshow(img, cmap="gray")
    ax.set_title(title)
    ax.axis("off")
plt.show()

## 2. Models (paper Tables 1 and 2)

In [ ]:
# Table 1: hyperparameter search space
PARAM_GRID = [
    {"kernel": ["linear"], "C": [0.1, 1, 10, 100]},
    {"kernel": ["rbf"], "C": [0.1, 1, 10, 100], "gamma": ["scale", 0.001, 0.0001]},
    {"kernel": ["poly"], "C": [0.1, 1, 10, 100], "degree": [2, 3, 4],
     "gamma": ["scale", 0.001, 0.0001]},
]

# Sec. 3.5: class weights as written in the paper (glioma, meningioma, no_tumor, pituitary)
PAPER_CLASS_WEIGHTS = {0: 0.69, 1: 0.69, 2: 0.90, 3: 0.60}


def paper_models():
    """The six SVMs compared in Table 2 (not trained yet)."""
    return {
        "SVM Custom Class Weight (Poly)": SVC(kernel="poly", degree=3, C=10,
                                              class_weight=PAPER_CLASS_WEIGHTS),
        "SVM Best Hyperparameters (Poly)": SVC(kernel="poly", degree=4, C=10),
        "SVM Polynomial": SVC(kernel="poly"),
        "SVM RBF": SVC(kernel="rbf"),
        "SVM Linear": SVC(kernel="linear"),
        "SVM Sigmoid": SVC(kernel="sigmoid"),
    }

## 3. Evaluation

Metrics: accuracy with a 95% bootstrap CI, weighted precision/recall/F1, macro one-vs-rest AUC, and per-class recall.

In [ ]:
def evaluate_model(name, model, X_test, y_test):
    """Predict on the test set, print a report, return a row of metrics (in %)."""
    y_pred = model.predict(X_test)
    precision, recall, f1, _ = precision_recall_fscore_support(
        y_test, y_pred, average="weighted", zero_division=0)
    class_recall = precision_recall_fscore_support(y_test, y_pred, average=None, zero_division=0)[1]

    # Macro AUC: each class vs the rest, from the SVM decision scores
    scores = model.decision_function(X_test)
    auc = np.mean([roc_auc_score(y_test == k, scores[:, k]) for k in range(len(CLASSES))])

    # 95% CI for accuracy: resample the test set 1000 times
    rng = np.random.default_rng(SEED)
    boot = []
    for _ in range(1000):
        idx = rng.integers(0, len(y_test), len(y_test))
        boot.append(accuracy_score(y_test[idx], y_pred[idx]))
    ci_low, ci_high = np.percentile(boot, [2.5, 97.5])

    print(f"\n=== {name} ===")
    print(classification_report(y_test, y_pred, target_names=CLASSES, digits=4, zero_division=0))

    row = {"model": name, "accuracy": accuracy_score(y_test, y_pred), "acc_ci_low": ci_low,
           "acc_ci_high": ci_high, "precision": precision, "recall": recall, "f1": f1}
    row.update({f"recall_{c}": r for c, r in zip(CLASSES, class_recall)})
    row = {k: (round(v * 100, 2) if k != "model" else v) for k, v in row.items()}
    row["macro_auc"] = round(auc, 4)
    return row, confusion_matrix(y_test, y_pred)

## 4. Experiment runner

In [ ]:
def run_experiment(use_clahe=False, skip_grid=False):
    # Step 1: images -> HOG features
    X_train, y_train = load_dataset(os.path.join(DATA_DIR, "Training"), use_clahe)
    X_test, y_test = load_dataset(os.path.join(DATA_DIR, "Testing"), use_clahe)
    print(f"Train {X_train.shape}, Test {X_test.shape}")

    # Step 2: PCA (fit on train only, so the test set never leaks into training)
    pca = PCA(0.98)
    X_train = pca.fit_transform(X_train)
    X_test = pca.transform(X_test)
    print(f"PCA components kept: {pca.n_components_}")

    # Step 3: train the models
    models = paper_models()
    if not skip_grid:
        search = GridSearchCV(SVC(), PARAM_GRID, cv=5, scoring="accuracy", n_jobs=-1)
        search.fit(X_train, y_train)
        print(f"GridSearch best: {search.best_params_}, CV acc {search.best_score_:.4f}")
        models = {"SVM GridSearch Best": search.best_estimator_, **models}
    for model in models.values():
        model.fit(X_train, y_train)

    # Step 4: evaluate, then plot all confusion matrices in one figure
    rows = []
    fig, axes = plt.subplots(2, 4, figsize=(16, 8))
    for ax, (name, model) in zip(axes.flat, models.items()):
        row, cm = evaluate_model(name, model, X_test, y_test)
        rows.append(row)
        ConfusionMatrixDisplay(cm).plot(ax=ax, cmap="Blues", colorbar=False)
        ax.set_title(name, fontsize=9)
    for ax in axes.flat[len(models):]:
        ax.axis("off")
    plt.tight_layout()
    plt.show()

    return pd.DataFrame(rows).set_index("model")

## 5. Baseline reproduction

In [ ]:
SKIP_GRID = False  # set True to skip the (slower) grid search

baseline = run_experiment(use_clahe=False, skip_grid=SKIP_GRID)
baseline

## 6. Our extension: CLAHE before resizing

Shrinking an MRI slice to 16×16 throws away most of its fine texture. CLAHE boosts local contrast at full resolution **before** downsampling, so tissue boundaries carry more weight in the small image that HOG sees. The rest of the pipeline is unchanged.

In [ ]:
clahe = run_experiment(use_clahe=True, skip_grid=SKIP_GRID)
clahe

## 7. Baseline vs CLAHE

In [ ]:
metrics = ["accuracy", "precision", "recall", "f1", "macro_auc"]
pd.concat({"Baseline": baseline[metrics], "CLAHE": clahe[metrics],
           "Δ (CLAHE − Baseline)": clahe[metrics] - baseline[metrics]}, axis=1)

In [ ]:
pd.DataFrame({"Baseline": baseline["accuracy"], "CLAHE": clahe["accuracy"]}).plot.bar(
    figsize=(10, 4), ylabel="Test accuracy (%)", rot=30)
plt.show()